# Resume API — Demo

This notebook demonstrates the `/jobs/{thread_id}/resume` endpoint, which picks up a failed or interrupted job **from its last LangGraph checkpoint** — no work is duplicated.

**Why this matters:** The 6-node pipeline can take 30–90 s end-to-end. If the server crashes after `generate_sections` (the most expensive step), resume restarts from `postprocess`, not from scratch.

---
### How checkpointing works

```
 serp_fetch → analyze_serp → build_outline → generate_sections → postprocess → validate_output
      ↓              ↓               ↓                ↓                ↓               ↓
  checkpoint     checkpoint      checkpoint       checkpoint      checkpoint      checkpoint
  (saved to checkpoints.db after every node)
```

When `resume` is called, LangGraph reads the latest checkpoint for that `thread_id` and re-enters the graph **at the next unexecuted node**. The `jobs.db` record is reset to `running` and the background task is re-queued.

---
### Two failure scenarios covered

| Scenario | Job status at resume time | How to trigger |
|---|---|---|
| **A — Server restart** | `running` (stale) | Stop server mid-job; restart; resume |
| **B — Node error** | `failed` | SerpAPI quota, LLM timeout, etc. |

---
## 1. Setup

> **Pre-requisite:** The API server must be running.
> ```bash
> cd seo_agent
> uvicorn main:app --reload --port 8000
> ```

In [9]:
import requests
import time

BASE_URL = "http://localhost:8000"

STAGES = [
    "serp_fetch",
    "analyze_serp",
    "build_outline",
    "generate_sections",
    "postprocess",
    "validate_output",
]

def poll_until_done(thread_id, *, timeout=300):
    """Poll /jobs/{thread_id} and print stage progress. Returns final job dict."""
    seen_stages = set()
    stage_start = time.time()
    deadline = time.time() + timeout
    poll_count = 0
    print("Polling — stages will appear as they complete:\n")

    while time.time() < deadline:
        resp = requests.get(f"{BASE_URL}/jobs/{thread_id}")
        resp.raise_for_status()
        data = resp.json()
        poll_count += 1

        stage = data.get("execution_stage")
        job_status = data["status"]

        if stage and stage not in seen_stages:
            idx = STAGES.index(stage) + 1 if stage in STAGES else "?"
            elapsed = time.time() - stage_start
            print(f"  [{idx}/{len(STAGES)}] ✓ {stage:<20}  {elapsed:5.1f}s  ({poll_count} poll{'s' if poll_count != 1 else ''})")
            seen_stages.add(stage)
            stage_start = time.time()
            poll_count = 0

        if job_status in ("completed", "failed"):
            final_icon = "✅" if job_status == "completed" else "❌"
            print(f"\nJob {job_status.upper()} {final_icon}")
            if data.get("error_message"):
                print(f"Error: {data['error_message']}")
            return data

        time.sleep(3)

    print("⚠️  Timed out waiting for job completion")
    return None

---
## 2. Scenario A — Resume after a server restart

**Goal:** Show that a job interrupted mid-pipeline picks up from its last checkpoint after the server comes back up.

**Playbook:**
1. Submit a job (`use_mock=True` so it runs fast enough to observe stages)
2. ⛔ Stop the server at any point while the job is running
3. ▶️ Restart the server
4. Check the job — it will be `status: running` with no further progress (stale)
5. Call `/resume` — the pipeline continues from the next unexecuted node

### Step 1 — Submit the job

In [10]:
payload = {
    "topic": "best productivity tools for remote teams",
    "target_word_count": 1200,
    "language": "en",
    "use_mock": True,   # mock SERP data — no SerpAPI key needed
}

resp = requests.post(f"{BASE_URL}/jobs", json=payload)
resp.raise_for_status()
job = resp.json()

thread_id = job["thread_id"]
print(f"✓ Job submitted")
print(f"  thread_id : {thread_id}")
print(f"  status    : {job['status']}")

✓ Job submitted
  thread_id : 18db4ee8-037f-4668-b07f-12fc20c68956
  status    : pending


### Step 2 — Interrupt the server

> **Action required (manual):**
> In your terminal, press `Ctrl+C` to stop `uvicorn` while the job is running.
> You can watch the stage progress by running the *optional* poll cell below before stopping.
>
> **When to stop:** Any time after job submission works. Stopping during `generate_sections` is the most dramatic — it's the longest node — but stopping immediately after submission also works fine.

In [11]:
# OPTIONAL — run this to watch stages tick by, then Ctrl+C the server in your terminal
# (interrupt this cell with the stop button once you've stopped the server)

for _ in range(20):
    resp = requests.get(f"{BASE_URL}/jobs/{thread_id}")
    data = resp.json()
    print(f"  status={data['status']:10s}  stage={data.get('execution_stage') or '—'}")
    if data["status"] in ("completed", "failed"):
        break
    time.sleep(3)

  status=running     stage=analyze_serp


KeyboardInterrupt: 

### Step 3 — Restart the server

> **Action required (manual):**
> ```bash
> cd seo_agent
> uvicorn main:app --reload --port 8000
> ```
> The server reloads `checkpoints.db` from disk — all prior node outputs are available.
> The `jobs.db` record still shows `status: running` (the server never updated it to `completed` or `failed`).

### Step 4 — Confirm the job is stale

In [13]:
resp = requests.get(f"{BASE_URL}/jobs/{thread_id}")
resp.raise_for_status()
job_state = resp.json()

print("Job status after restart:")
print(f"  status          : {job_state['status']}")
print(f"  execution_stage : {job_state.get('execution_stage')}")
print(f"  error_message   : {job_state.get('error_message')}")

if job_state["status"] == "running":
    last_stage = job_state.get('execution_stage') or 'none'
    if last_stage in STAGES:
        next_idx = STAGES.index(last_stage) + 1
        next_stage = STAGES[next_idx] if next_idx < len(STAGES) else "(end)"
    else:
        next_stage = STAGES[0]
    print(f"\n⚠️  Job is stale — server restarted while it was in-flight.")
    print(f"   Last known stage: {last_stage}")
    print(f"   The pipeline will resume from the NEXT node: {next_stage}")

Job status after restart:
  status          : running
  execution_stage : build_outline
  error_message   : None

⚠️  Job is stale — server restarted while it was in-flight.
   Last known stage: build_outline
   The pipeline will resume from the NEXT node: generate_sections


### Step 5 — Resume

In [14]:
resp = requests.post(f"{BASE_URL}/jobs/{thread_id}/resume")
resp.raise_for_status()
resume_resp = resp.json()

print(f"✓ Resume accepted")
print(f"  thread_id : {resume_resp['thread_id']}")
print(f"  status    : {resume_resp['status']}")

✓ Resume accepted
  thread_id : 18db4ee8-037f-4668-b07f-12fc20c68956
  status    : running


### Step 6 — Poll to completion

> Notice that only stages **after** the interruption point appear — completed nodes are not re-run.

In [15]:
final = poll_until_done(thread_id)

Polling — stages will appear as they complete:

  [1/6] ✓ serp_fetch              2.0s  (1 poll)
  [2/6] ✓ analyze_serp            5.1s  (1 poll)
  [3/6] ✓ build_outline          30.2s  (6 polls)


KeyboardInterrupt: 

> **Before resuming:** fix the root cause (add the API key, wait out a rate-limit, etc.).
> Then call `/resume` — the pipeline re-enters at the failed node with the same `thread_id`.
>
> **Note:** The resume endpoint always reconstructs the `JobRequest` with `use_mock=False`.
> If you originally submitted with `use_mock=True` and want to retry with mock data,
> submit a fresh job instead.

In [2]:
# Fix the underlying issue first, then call resume
resp = requests.post(f"{BASE_URL}/jobs/{failed_thread_id}/resume")
resp.raise_for_status()
print(f"✓ Resume accepted — pipeline re-queued")
print(f"  status : {resp.json()['status']}")

NameError: name 'requests' is not defined

In [1]:
final = poll_until_done(failed_thread_id)

NameError: name 'poll_until_done' is not defined